# 🤖 Заняття 8. Побудова мультиагентної системи
---

У цьому воркшопі ми побудуємо **персонального AI-асистента** з архітектурою supervisor + sub-agents, використовуючи `create_agent` з LangChain 1.x та LangGraph runtime.

Система координуватиме трьох спеціалізованих агентів:
- 📅 **Calendar Agent** — планування зустрічей (Google Calendar API)
- 📧 **Email Agent** — надсилання та пошук листів (Gmail API)
- 🔍 **Research Agent** — веб-пошук та RAG по внутрішній базі знань

**Базовий приклад:** [Personal Assistant with Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents-personal-assistant)


## 🎯 Очікувані результати

Після цього заняття ви зможете:

1. Будувати **supervisor + sub-agents** архітектуру з використанням `create_agent`
2. Обгортати суб-агенти як **tools** для supervisor'а та контролювати інформаційний потік
3. Інтегрувати **реальні інструменти** (API) в агентну систему
4. Реалізовувати **Human-in-the-Loop** з `HumanInTheLoopMiddleware`
5. Застосовувати **middleware** для агентних систем


---

# Блок 1. Контекст і архітектура

---

## 1.1. Recap: що ми вже знаємо

Перед тим як будувати, переконаємось що ключові концепти з попередніх занять свіжі в пам'яті.

**Заняття 5 — RAG-системи:**
- RAG як патерн: retrieval → augmentation → generation замість запихування всього в контекст
- Ingestion pipeline: document loaders → chunking → embeddings → vector store
- Hybrid search (dense + sparse), reranking для покращення якості retrieval
- **Agentic RAG** — агент сам вирішує коли і як шукати, робить query rewriting та self-reflection

**Заняття 6 — Інструменти оркестрації:**
- Чотири парадигми оркестрації: графова (LangGraph), рольова (CrewAI), розмовна (Microsoft Agent Framework)
- LangGraph: state → nodes → edges, checkpointer для persistence та Human-in-the-Loop
- Вибір фреймворку залежить від задачі: складні workflows → LangGraph, рольова колаборація → CrewAI

**Заняття 7 — Патерни мультиагентної взаємодії:**
- Таксономія патернів: sequential, parallel, routing, orchestrator-workers, evaluator-optimizer (Anthropic)
- Ролі в AI Teams: Planner, Executor/Coder, Critic/Reviewer, Evaluator
- Кооперативна vs конкурентна vs ієрархічна взаємодія
- Supervisor pattern — централізований координатор, який делегує задачі субагентам
- Hierarchical agent teams — supervisor of supervisors
- **Coordination tax** — мультиагентність додає складності, використовувати тільки коли є реальна потреба

## 1.2. Архітектура системи

Наша система має **три шари** (layered architecture):

| Шар | Компоненти | Відповідальність |
|-----|-----------|-----------------|
| **Top: Supervisor** | Персональний асистент | Роутинг, координація, синтез результатів |
| **Mid: Sub-Agents** | Calendar, Email, Research | Розуміння природної мови → виклик правильних API → відповідь |
| **Bottom: Tools** | Google Cal, Gmail, Tavily, FAISS | API з точним форматом вхідних даних |

Ключова ідея: **кожен шар абстрагує складність нижнього шару** для шару вище.


![Personal Assistant — Supervisor Architecture](images/architecture.png)

> 🔑 Supervisor бачить тільки **високорівневі capabilities** (`schedule_event`, `manage_email`, `research`), а не конкретні API (`create_calendar_event`, `send_email`). Це зменшує когнітивне навантаження на модель та покращує якість роутингу.


## 1.3. Ключові інструменти LangChain 1.x для агентів

Сьогодні ми будуватимемо систему на LangChain 1.x та LangGraph 1.x. Ось основні концепти, які нам знадобляться:

| Концепт | Що це і навіщо                                                                                                                                 |
|---------|------------------------------------------------------------------------------------------------------------------------------------------------|
| **`create_agent`** | Головна функція для створення агента — приймає модель, tools і prompt, а під капотом будує агента з повним agent loop (reason → act → observe) |
| **Middleware** | "Прошарки", які додають поведінку до агента без зміни його коду: Human-in-the-Loop, summarization довгих розмов тощо            |
| **`ToolRuntime`** | Дає tool-функціям доступ до стану агента — наприклад, tool може прочитати попередні повідомлення або metadata                                  |
| **`init_chat_model`** | Універсальна ініціалізація моделі: `"openai:gpt-5.4"`, `"anthropic:claude-sonnet-4-6"` — легко змінити провайдера одним рядком                 |
| **Checkpointer** | Зберігає стан розмови між викликами (`InMemorySaver` для dev, PostgreSQL для prod). Обов'язковий для Human-in-the-Loop                         |
| **Standard content blocks** | Уніфікований формат відповідей від різних LLM — не треба писати окремий парсинг для OpenAI та Anthropic                                        |

---

# Блок 2. Воркшоп — будуємо систему крок за кроком

---

## 2.1. Setup та конфігурація

Почнемо з встановлення всіх необхідних бібліотек та конфігурації API-ключів.


In [1]:
!pip install -q langchain langgraph langchain-openai langchain-community tavily-python faiss-cpu langchain-text-splitters google-api-python-client google-auth-httplib2 google-auth-oauthlib nltk scipy scikit-learn pandas "numpy<2"

import warnings
warnings.filterwarnings("ignore")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### 🔐 Як отримати `client_secret.json` для Google API

1. Перейдіть на [Google Cloud Console](https://console.cloud.google.com/)
2. Створіть новий проєкт (або оберіть існуючий)
3. **APIs & Services → Library** — увімкніть **Google Calendar API** та **Gmail API**
4. **APIs & Services → OAuth consent screen** — оберіть **External**, заповніть назву додатку, додайте свій email як test user
5. **APIs & Services → Credentials → Create Credentials → OAuth client ID** — тип **Desktop app**
6. Натисніть **Download JSON** — це і є ваш `client_secret.json`
7. Покладіть файл у кореневу папку проєкту (поруч з ноутбуком)

> ⚠️ **Не комітьте `client_secret.json` та `token.json` в git!**

In [3]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key: ")

# --- Demo config ---
DEMO_EMAIL = "shanin.vladyslav@gmail.com"

# --- Google OAuth2 Setup ---
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

GOOGLE_SCOPES = [
    "https://www.googleapis.com/auth/calendar",
    "https://www.googleapis.com/auth/gmail.send",
    "https://www.googleapis.com/auth/gmail.readonly",
]

def get_google_credentials(
    client_secret_path: str = "client_secret.json",
    token_path: str = "token.json",
) -> Credentials:
    """Authenticate with Google OAuth2. Opens browser on first run."""
    creds = None
    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, GOOGLE_SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secret_path, GOOGLE_SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_path, "w") as f:
            f.write(creds.to_json())
    return creds

google_creds = get_google_credentials()
calendar_service = build("calendar", "v3", credentials=google_creds)
gmail_service = build("gmail", "v1", credentials=google_creds)

print("✅ API keys configured")
print("✅ Google Calendar & Gmail APIs authenticated")

✅ API keys configured
✅ Google Calendar & Gmail APIs authenticated


> 🔑 **Google OAuth2:** При першому запуску відкриється вікно браузера для авторизації. Після цього `token.json` зберігає сесію, і наступні запуски не потребують повторної авторизації.

In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.4")

# Test the model
response = model.invoke("Say 'hello'")
print(response.text)

hello


---

## 2.2. Визначення низькорівневих інструментів (Tools)

Інструменти — це **найнижчий шар** нашої системи. Вони мають жорсткий інтерфейс з точною типізацією параметрів. Tools працюють з реальними Google Calendar та Gmail API через OAuth2.

### 📅 Calendar Tools


In [ ]:
# Calendar tools — Google Calendar API
from langchain.tools import tool
from datetime import datetime, timedelta
from typing import Optional
from pprint import pprint
import json


@tool
def get_available_time_slots(date: str, attendees: Optional[list[str]] = None) -> str:
    """Check calendar availability for given attendees on a specific date.

    Args:
        attendees: List of email addresses to check availability for
        date: Date in ISO format (e.g., '2026-04-01')
    """
    if attendees is None:
        attendees = ["primary"]
    body = {
        "timeMin": f"{date}T00:00:00+03:00",
        "timeMax": f"{date}T23:59:59+03:00",
        "timeZone": "Europe/Kyiv",
        "items": [{"id": email} for email in attendees],
    }
    result = calendar_service.freebusy().query(body=body).execute()
    return json.dumps(result["calendars"], default=str)


@tool
def create_calendar_event(
    title: str, start_time: str, end_time: str, attendees: list[str], location: str = ""
) -> str:
    """Create a Google Calendar event.

    Args:
        title: Event title/subject
        start_time: Start time in ISO format (e.g., '2026-04-01T14:00:00')
        end_time: End time in ISO format (e.g., '2026-04-01T15:00:00')
        attendees: List of attendee email addresses
        location: Optional meeting location or video call link
    """
    event = calendar_service.events().insert(
        calendarId="primary",
        sendUpdates="all",
        body={
            "summary": title,
            "location": location,
            "start": {"dateTime": start_time, "timeZone": "Europe/Kyiv"},
            "end": {"dateTime": end_time, "timeZone": "Europe/Kyiv"},
            "attendees": [{"email": e} for e in attendees],
        },
    ).execute()
    return json.dumps({"status": "confirmed", "event_id": event["id"], "link": event.get("htmlLink", "")})


# Quick test — check availability
print("📅 Available slots:")
pprint(json.loads(get_available_time_slots.invoke({"attendees": ["primary"], "date": "2026-04-02"})))

### 📧 Email Tools


In [ ]:
# Email tools — Gmail API
import base64
from email.mime.text import MIMEText


@tool
def send_email(
    to: list[str],
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via Gmail API.

    Args:
        to: List of recipient email addresses
        subject: Email subject line
        body: Email body text (plain text)
        cc: Optional list of CC email addresses

    Returns:
        Confirmation with message details
    """
    message = MIMEText(body)
    message["to"] = ", ".join(to)
    message["subject"] = subject
    if cc:
        message["cc"] = ", ".join(cc)

    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()
    result = gmail_service.users().messages().send(
        userId="me", body={"raw": raw}
    ).execute()

    return json.dumps({
        "status": "sent",
        "message_id": result["id"],
        "thread_id": result.get("threadId", ""),
        "to": to,
        "cc": cc,
        "subject": subject,
    })


@tool
def search_emails(
    query: str,
    max_results: int = 5
) -> str:
    """Search emails in Gmail by query string.

    Args:
        query: Gmail search query (supports operators like from:, to:, subject:, after:)
        max_results: Maximum number of results to return (default: 5)

    Returns:
        JSON with matching email summaries
    """
    response = gmail_service.users().messages().list(
        userId="me", q=query, maxResults=max_results
    ).execute()

    messages = response.get("messages", [])
    results = []
    for msg in messages:
        detail = gmail_service.users().messages().get(
            userId="me", id=msg["id"], format="metadata"
        ).execute()
        headers = {h["name"]: h["value"] for h in detail["payload"]["headers"]}
        results.append({
            "from": headers.get("From", ""),
            "subject": headers.get("Subject", ""),
            "date": headers.get("Date", ""),
            "snippet": detail.get("snippet", ""),
        })

    return json.dumps({"results": results, "total_matches": len(results)})


# Quick test — search recent emails
print("📧 Recent emails:")
pprint(json.loads(search_emails.invoke({"query": "newer_than:7d", "max_results": 3})))

### 🔍 Research Tools: Tavily Web Search + RAG


In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_search = TavilySearchResults(
    max_results=3,
    search_depth="basic",
)

# Test Tavily (requires TAVILY_API_KEY)
try:
    results = tavily_search.invoke("LangGraph multi-agent 2026")
    print(f"🔍 Tavily returned {len(results)} results")
    for r in results[:2]:
        print(f"  - {r.get('url', 'N/A')}")
except Exception as e:
    print(f"⚠️ Tavily not available: {e}")

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Simulate internal company documents
company_docs = [
    Document(
        page_content="""Company Vacation Policy (Effective April 2026):
        All employees are entitled to 25 days of paid vacation per year.
        Vacation requests must be submitted at least 2 weeks in advance.
        Maximum consecutive vacation: 10 business days without VP approval.
        Unused vacation days can be carried over (max 5 days) to next year.""",
        metadata={"source": "hr-policies/vacation.md", "updated": "2026-03-01"}
    ),
    Document(
        page_content="""Meeting Room Booking Guidelines:
        Large conference rooms (10+ people): book via calendar, max 2 hours.
        Small huddle rooms (2-4 people): first-come-first-serve, no booking needed.
        All-hands room: requires VP approval for bookings.
        Video conferencing equipment is available in all large rooms.""",
        metadata={"source": "facilities/rooms.md", "updated": "2026-02-15"}
    ),
    Document(
        page_content="""Team Contact Directory:
        Engineering Lead: alice@company.com (Alice Chen)
        Product Manager: bob@company.com (Bob Wilson)
        Design Lead: carol@company.com (Carol Davis)
        CTO: david@company.com (David Kim)
        HR Manager: elena@company.com (Elena Petrova)
        Full directory available at: intranet.company.com/people""",
        metadata={"source": "team/contacts.md", "updated": "2026-03-20"}
    ),
    Document(
        page_content="""Q1 2026 OKR Summary:
        Objective 1: Launch v2.0 of the platform — 85% complete.
        Key Result: Reduce API latency by 40% — achieved 38%.
        Key Result: Onboard 50 enterprise clients — achieved 47.
        Objective 2: Improve team satisfaction score to 4.5/5 — current: 4.3.
        Next review: April 15, 2026.""",
        metadata={"source": "strategy/okr-q1-2026.md", "updated": "2026-03-30"}
    ),
    Document(
        page_content="""IT Security Guidelines:
        All employees must use 2FA for all company accounts.
        Password rotation: every 90 days, minimum 14 characters.
        VPN required for accessing internal services from outside the office.
        Report suspicious emails to security@company.com immediately.
        Do NOT share API keys in Slack, email, or public repositories.""",
        metadata={"source": "it/security.md", "updated": "2026-01-10"}
    ),
]

# Build vector index
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(company_docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(splits, embeddings)

print(f"✅ Knowledge base built: {len(splits)} chunks from {len(company_docs)} documents")

✅ Knowledge base built: 5 chunks from 5 documents


In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search internal company knowledge base using semantic search.

    Use this for questions about company policies, team contacts,
    OKRs, IT security, meeting room guidelines, and other internal information.

    Args:
        query: Natural language search query

    Returns:
        Relevant excerpts from internal documents with source attribution
    """
    docs = vectorstore.similarity_search(query, k=2)
    results = []
    for doc in docs:
        results.append({
            "content": doc.page_content.strip(),
            "source": doc.metadata.get("source", "unknown"),
            "updated": doc.metadata.get("updated", "unknown")
        })
    return json.dumps(results, ensure_ascii=False)


print("🔍 Knowledge base search: 'vacation policy'")
result = search_knowledge_base.invoke({"query": "How many vacation days do we get?"})
print(json.dumps(json.loads(result), indent=2, ensure_ascii=False))

> 🔑 **Tool docstrings — це промпт для моделі!**
>
> Коли агент вирішує який інструмент використати, він читає саме docstring. Погано написаний docstring = погано обраний інструмент. Зверніть увагу:
> - **Чітко описуйте**, коли саме використовувати цей інструмент
> - **Типізуйте параметри** — це критично для tool calling
> - **Описуйте формат відповіді** — агент має розуміти що він отримає


---

## 2.3. Створення спеціалізованих суб-агентів

Тепер обгорнемо наші tools у **спеціалізованих агентів**. Кожен агент:
- Має свій **system prompt** з чіткою роллю та правилами
- Має доступ тільки до **своїх** інструментів
- Приймає запит **природною мовою** і повертає відповідь природною мовою

Це **середній шар** нашої архітектури.


### 📅 Calendar Agent


In [10]:
from langchain.agents import create_agent

CALENDAR_AGENT_PROMPT = """You are a calendar scheduling assistant.

Your responsibilities:
- Parse natural language scheduling requests into ISO datetime formats
- Check availability before creating events
- Handle timezone awareness (default timezone: Europe/Kyiv)
- Resolve ambiguous dates ("next Tuesday", "tomorrow afternoon", "end of the week")

Rules:
- ALWAYS check availability with get_available_time_slots before creating an event
- If no suitable slot exists, report back — do NOT create conflicting events
- Always confirm what was scheduled in your final response
- Include all attendees mentioned in the original request
- For date parsing: today is {today}
""".format(today=datetime.now().strftime("%A, %B %d, %Y"))

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

print("✅ Calendar Agent created")

✅ Calendar Agent created


In [ ]:
# Test Calendar Agent in isolation
for step in calendar_agent.stream({
    "messages": [
        {
            "role": "user",
            "content": f"Schedule a team standup tomorrow at 9am for 30 minutes with {DEMO_EMAIL}"
        }
    ]
}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

### 📧 Email Agent


In [12]:
# Email Agent — communication specialist

EMAIL_AGENT_PROMPT = """You are a professional email assistant.

Your responsibilities:
- Compose clear, professional emails based on natural language requests
- Extract and validate recipient information
- Generate appropriate subject lines
- Match tone to context (formal for external, friendly for internal)
- Search past emails when asked about previous communication

Rules:
- If recipients aren't specified by email address, use search_emails to find them
- Always confirm what was sent in your final response
- Keep emails concise and actionable
- Include a greeting and sign-off appropriate to the context
"""

email_agent = create_agent(
    model,
    tools=[send_email, search_emails],
    system_prompt=EMAIL_AGENT_PROMPT,
)

print("✅ Email Agent created")

✅ Email Agent created


In [ ]:
# Test Email Agent in isolation

for step in email_agent.stream({
    "messages": [
        {
            "role": "user",
            "content": f"Send a reminder to the design team ({DEMO_EMAIL}) about reviewing the new mockups before Friday"
        }
    ]
}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

### 🔍 Research Agent


In [14]:
# Research Agent — information specialist

RESEARCH_AGENT_PROMPT = """You are a research assistant with access to web search and an internal knowledge base.

Your responsibilities:
- Search the web for current, external information using tavily_search
- Search the internal knowledge base for company-specific information using search_knowledge_base
- Synthesize information from multiple sources into concise summaries
- Provide source attribution for all claims

Rules:
- PREFER internal knowledge base for company-related questions (policies, contacts, OKRs)
- USE web search for external/current information (market data, news, technology updates)
- If sources conflict, note the discrepancy
- Always cite which source your information came from
- Keep summaries factual and concise
"""

research_agent = create_agent(
    model,
    tools=[tavily_search, search_knowledge_base],
    system_prompt=RESEARCH_AGENT_PROMPT,
)

print("✅ Research Agent created")

✅ Research Agent created


In [ ]:
for step in research_agent.stream({
    "messages": [
        {"role": "user", "content": "What is our company's vacation policy? How many days do we get?"}
    ]
}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

> 🔑 **Тестуйте кожного агента окремо!**
>
> Перш ніж інтегрувати агентів у supervisor, переконайтесь що кожен з них працює коректно ізольовано. Це значно спрощує дебаг — якщо щось не працює в supervisor'і, ви вже знаєте що проблема не в суб-агенті.


---

## 2.4. Обгортка суб-агентів у tools для Supervisor

Це **ключовий архітектурний крок**. Ми обгортаємо кожного суб-агента у tool-функцію, яку зможе використовувати supervisor. Supervisor бачитиме тільки:
- `schedule_event` — а не `create_calendar_event` + `get_available_time_slots`
- `manage_email` — а не `send_email` + `search_emails`
- `research` — а не `tavily_search` + `search_knowledge_base`

Це **абстракція**: supervisor працює на рівні capabilities, а не API.


In [16]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to:
    - Create or schedule a meeting/event
    - Check availability for a specific date/time
    - Set up recurring meetings

    Input: Natural language scheduling request
    (e.g., 'meeting with design team next Tuesday at 2pm for 1 hour')
    """
    result = calendar_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].text


@tool
def manage_email(request: str) -> str:
    """Send or search emails using natural language.

    Use this when the user wants to:
    - Send a notification, reminder, or any email
    - Search for past emails or conversations
    - Draft and send a follow-up message

    Input: Natural language email request
    (e.g., 'send the team a reminder about the deadline')
    """
    result = email_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].text


@tool
def research(request: str) -> str:
    """Research topics using web search and internal knowledge base.

    Use this when the user wants to:
    - Find current information from the web
    - Look up company policies, contacts, or OKRs
    - Get summaries of internal documentation
    - Research a topic before a meeting or decision

    Input: Natural language research question
    (e.g., 'what is our vacation policy?' or 'latest news about AI regulation')
    """
    result = research_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].text


print("✅ Sub-agent tools created: schedule_event, manage_email, research")

✅ Sub-agent tools created: schedule_event, manage_email, research


> ⚠️ **Важливо:** Ми повертаємо тільки **фінальну відповідь** суб-агента (`result["messages"][-1].text`). Supervisor не бачить проміжних reasoning кроків чи tool calls суб-агента. Це:
> - Зменшує токени (менше контексту для supervisor'а)
> - Спрощує роботу supervisor'а (він бачить тільки результат)
> - Зберігає **separation of concerns** між шарами


---

## 2.5. Створення Supervisor Agent

Supervisor — це **верхній шар** системи. Він:
- Отримує запит користувача
- Вирішує, яких суб-агентів залучити
- Координує їх роботу (послідовно або паралельно)
- Синтезує результати у фінальну відповідь


In [ ]:
# Create the Supervisor Agent
from datetime import datetime

SUPERVISOR_PROMPT = f"""You are a helpful personal assistant that coordinates specialized agents.

Available capabilities:
- schedule_event: Schedule calendar events, check availability
- manage_email: Send emails, search past correspondence
- research: Search the web and internal knowledge base

Coordination rules:
- Break down complex requests into appropriate tool calls
- When tasks are independent, call multiple tools in parallel
- When tasks depend on each other, call them sequentially
- Synthesize results from multiple tools into a clear, unified response
- If a tool reports an error, inform the user and suggest alternatives
- Never fabricate information — always rely on your tools for facts
- Be concise but thorough in your final response

Keep in mind that the datetime now is {datetime.now().isoformat()}
"""

supervisor = create_agent(
    model,
    tools=[schedule_event, manage_email, research],
    system_prompt=SUPERVISOR_PROMPT,
)

print("✅ Supervisor Agent created")
print("\n🏗️ System architecture:")
print("  Supervisor")
print("  ├── schedule_event  → Calendar Agent → [create_calendar_event, get_available_time_slots]")
print("  ├── manage_email    → Email Agent    → [send_email, search_emails]")
print("  └── research        → Research Agent → [tavily_search, search_knowledge_base]")

### Тест 1: Простий single-domain запит


In [ ]:
# Test 1: Simple single-domain request
for step in supervisor.stream(
    {"messages": [{"role": "user", "content": "What is our company's vacation policy?"}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

### Тест 2: Складний multi-domain запит

Тепер спробуємо запит, який вимагає координації **кількох агентів** одночасно. Supervisor має сам розібратися, яких агентів залучити і в якому порядку.


In [ ]:
# Test 2: Complex multi-domain request (requires 2-3 agents)

query = (
    f"Schedule a meeting with the design team ({DEMO_EMAIL}) "
    "next Wednesday at 2pm for 1 hour to discuss Q1 OKR results, "
    "and send them an email with a summary of our Q1 OKRs as context."
)

for step in supervisor.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

> 🔑 **Зверніть увагу** на порядок виконання: supervisor може викликати `research` (для OKR даних) та `schedule_event` **паралельно**, а потім `manage_email` **послідовно** (бо потрібні результати research'у). Або може обрати іншу стратегію — це залежить від моделі.


---

## 2.6. Human-in-the-Loop (HITL)

У production системі ми не хочемо, щоб агент надсилав листи чи створював зустрічі **без підтвердження** користувача. **Human-in-the-Loop** — це механізм, який:

1. **Зупиняє** виконання перед критичною дією (write-операцією)
2. **Показує** користувачу що саме агент хоче зробити
3. **Чекає** рішення: `approve` / `edit` / `reject`
4. **Продовжує** або скасовує виконання

Для цього нам потрібні дві речі:
- `HumanInTheLoopMiddleware` — middleware, що перехоплює tool calls
- `InMemorySaver` **checkpointer** — зберігає стан агента між interrupt і resume


In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# Calendar Agent with HITL on create (but NOT on availability check)
calendar_agent_hitl = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "create_calendar_event": True,       # ← REQUIRES approval
                "get_available_time_slots": False,   # ← auto-approved (read-only)
            },
            description_prefix="📅 Calendar event pending approval",
        ),
    ],
)

# Email Agent with HITL on send (but NOT on search)
email_agent_hitl = create_agent(
    model,
    tools=[send_email, search_emails],
    system_prompt=EMAIL_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {
                    "allowed_decisions": ["approve", "reject"],
                },
                "search_emails": False,  # read-only, no approval needed
            },
            description_prefix="📧 Outbound email pending approval",
        ),
    ],
)

# Research Agent — no HITL needed (read-only operations)
# research_agent stays as-is

print("✅ Sub-agents recreated with HITL middleware")
print("  📅 Calendar: interrupt on create_calendar_event")
print("  📧 Email:    interrupt on send_email")
print("  🔍 Research: no interrupts (read-only)")

> 🔑 **Принцип: Read vs Write**
>
> | Операція | Тип | HITL? |
> |----------|-----|-------|
> | `get_available_time_slots` | Read | ❌ Ні |
> | `search_emails` | Read | ❌ Ні |
> | `search_knowledge_base` | Read | ❌ Ні |
> | `tavily_search` | Read | ❌ Ні |
> | `create_calendar_event` | **Write** | ✅ **Так** |
> | `send_email` | **Write** | ✅ **Так** |


In [21]:
@tool
def schedule_event_hitl(request: str) -> str:
    """Schedule calendar events using natural language.
    Use when the user wants to create/schedule meetings or check availability.
    """
    result = calendar_agent_hitl.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].text


@tool
def manage_email_hitl(request: str) -> str:
    """Send or search emails using natural language.
    Use when the user wants to send notifications or search past emails.
    """
    result = email_agent_hitl.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].text


# Supervisor with checkpointer (REQUIRED for HITL)
supervisor_hitl = create_agent(
    model,
    tools=[schedule_event_hitl, manage_email_hitl, research],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver(),  # ← Required to save state during interrupts
)

print("✅ Supervisor with HITL ready")

✅ Supervisor with HITL ready


In [ ]:
from langgraph.types import Command, Interrupt

def print_messages(update):
    """Print messages from a stream update if present."""
    if isinstance(update, dict):
        for message in update.get("messages", []):
            message.pretty_print()

def print_interrupt(interrupt):
    """Display interrupt details for user review."""
    print(f"\n{'=' * 60}")
    print(f"⏸️  ACTION REQUIRES APPROVAL")
    print(f"{'=' * 60}")
    for request in interrupt.value.get("action_requests", []):
        print(f"  Tool:  {request.get('action', 'N/A')}")
        print(f"  Args:  {json.dumps(request.get('args', {}), indent=2, ensure_ascii=False)}")
    print()

def resume_graph(interrupt_id, decision_type):
    """Resume the graph after an interrupt with the given decision."""
    resume = {interrupt_id: {"decisions": [{"type": decision_type}]}}
    for step in supervisor_hitl.stream(Command(resume=resume), config):
        for update in step.values():
            print_messages(update)

config = {"configurable": {"thread_id": "hitl-demo"}}
query = f"Schedule a meeting with Vlad ({DEMO_EMAIL}) on 2026-04-07 at 2pm for 1 hour"

for step in supervisor_hitl.stream(
    {"messages": [{"role": "user", "content": query}]},
    config,
):
    for update in step.values():
        if isinstance(update, tuple) and isinstance(update[0], Interrupt):
            interrupt = update[0]
            print_interrupt(interrupt)

            decision = input("👉 approve / reject: ").strip().lower()
            approved = decision == "approve"

            print("\n✅ Approved! Resuming...\n" if approved else "\n❌ Rejected.")
            resume_graph(interrupt.id, "approve" if approved else "reject")
        else:
            print_messages(update)

> 💡 `input()` у Jupyter створює інтерактивне текстове поле прямо під ячейкою. Введіть `approve` щоб підтвердити дію, або `reject` щоб скасувати.
>
> В production додатку замість `input()` ви б використовували UI (кнопки в Slack, веб-інтерфейс тощо), але механізм interrupt/resume залишається тим самим.

> 💡 **Edit flow:** В реальному додатку ви можете не тільки approve/reject, а й **edit** параметри tool call перед виконанням. Наприклад, змінити тему листа або час зустрічі:
>
> ```python
> edited_action = interrupt_.value["action_requests"][0].copy()
> edited_action["args"]["subject"] = "Updated subject line"
> resume[interrupt_.id] = {
>     "decisions": [{"type": "edit", "edited_action": edited_action}]
> }
> ```


---

## 2.7. Розширення: контроль інформаційного потоку

За замовчуванням суб-агенти отримують **тільки** текст запиту від supervisor'а. Але інколи їм потрібен ширший контекст: оригінальне повідомлення користувача, попередня переписка, або результати інших агентів.

**`ToolRuntime`** дозволяє отримати доступ до повного стану supervisor'а зсередини tool-функції.


In [24]:
# Advanced: Pass full conversation context to sub-agents using ToolRuntime
from langchain.tools import tool, ToolRuntime


@tool
def schedule_event_ctx(request: str, runtime: ToolRuntime) -> str:
    """Schedule calendar events with full conversation context."""
    # Get the original user message from supervisor's state
    original_user_message = next(
        (msg for msg in runtime.state["messages"] if msg.type == "human"),
        None
    )

    # Enrich the sub-agent's context
    if original_user_message:
        enriched_prompt = (
            f"Original user request: {original_user_message.text}\n\n"
            f"Your specific task: {request}"
        )
    else:
        enriched_prompt = request

    result = calendar_agent.invoke(
        {"messages": [{"role": "user", "content": enriched_prompt}]}
    )
    return result["messages"][-1].text


print("✅ Context-aware tool created")
print()
print("Коли це корисно?")
print('  • Користувач каже: "schedule IT for the same time tomorrow"')
print('    → Без контексту суб-агент не знає що таке "it"')
print('  • Supervisor вже обговорював деталі з користувачем')
print('    → Суб-агент бачить повну історію розмови')

✅ Context-aware tool created

Коли це корисно?
  • Користувач каже: "schedule IT for the same time tomorrow"
    → Без контексту суб-агент не знає що таке "it"
  • Supervisor вже обговорював деталі з користувачем
    → Суб-агент бачить повну історію розмови


> ⚠️ **Trade-off:** Передача повного контексту суб-агенту збільшує кількість токенів і може включати нерелевантну інформацію. Використовуйте `ToolRuntime` **вибірково** — тільки коли контекст дійсно потрібен для правильного виконання задачі.


---

# Блок 3. Production Concerns

---

## 3.1. Middleware для production

LangChain 1.x надає набір **composable middleware** — модульних компонентів, які додаються до агента без зміни core logic. Ми вже використали `HumanInTheLoopMiddleware`. Розглянемо інші:


In [25]:
# SummarizationMiddleware — manages long conversations
from langchain.agents.middleware import SummarizationMiddleware

# Example: supervisor with summarization for long conversations
supervisor_prod = create_agent(
    model,
    tools=[schedule_event, manage_email, research],
    system_prompt=SUPERVISOR_PROMPT,
    middleware=[
        SummarizationMiddleware(
            model="gpt-5.4-mini",            # use cheaper model for summarization
            trigger=("tokens", 4000),        # trigger when context > 4000 tokens
            keep=("messages", 10),           # keep last 10 messages intact
        ),
    ],
)

print("✅ Supervisor with SummarizationMiddleware created")
print()
print("How it works:")
print("  1. Agent runs normally until context approaches 4000 tokens")
print("  2. Middleware triggers: older messages are summarized by gpt-5.4-mini")
print("  3. Summary replaces old messages, last 10 messages are kept as-is")
print("  4. Agent continues with compressed context — no information loss")

✅ Supervisor with SummarizationMiddleware created

How it works:
  1. Agent runs normally until context approaches 4000 tokens
  2. Middleware triggers: older messages are summarized by gpt-5.4-mini
  3. Summary replaces old messages, last 10 messages are kept as-is
  4. Agent continues with compressed context — no information loss


### Огляд всіх доступних middleware

| Middleware | Призначення | Коли використовувати |
|-----------|-------------|---------------------|
| `HumanInTheLoopMiddleware` | Підтвердження write-операцій | Надсилання листів, створення подій, зміни |
| `SummarizationMiddleware` | Стиснення довгих розмов | Довгі сесії, context window management |
| `ModelFallbackMiddleware` | Fallback на іншу модель | Якщо основна модель недоступна |
| `PIIMiddleware` | Детекція/маскування PII | Коли агент обробляє персональні дані |
| `ModelCallLimitMiddleware` | Обмеження кількості LLM-викликів | Захист від infinite loops, контроль costs |
| `ToolCallLimitMiddleware` | Обмеження кількості tool-викликів | Захист від run-away tools |

> 💡 Middleware **композуються** — ви можете комбінувати кілька в одному агенті. Порядок має значення: вони виконуються як pipeline.


---

## 3.2. Error handling у tools

У production tool calls **будуть** падати: API таймаути, rate limits, невалідні параметри. Ключовий принцип:

> 🔑 **Tool повертає помилку як текст, а не кидає exception.** Це дозволяє агенту обробити помилку і повідомити користувача.


In [26]:
# Error handling pattern for production tools

@tool
def send_email_safe(
    to: list[str], subject: str, body: str, cc: list[str] = []
) -> str:
    """Send an email via Gmail API with error handling."""
    try:
        # Validate inputs
        if not to:
            return "ERROR: No recipients specified. Please provide at least one email address."

        for email in to:
            if "@" not in email:
                return f"ERROR: Invalid email address: '{email}'. Please provide a valid email."

        message = MIMEText(body)
        message["to"] = ", ".join(to)
        message["subject"] = subject
        if cc:
            message["cc"] = ", ".join(cc)

        raw = base64.urlsafe_b64encode(message.as_bytes()).decode()
        result = gmail_service.users().messages().send(
            userId="me", body={"raw": raw}
        ).execute()

        return json.dumps({
            "status": "sent",
            "message_id": result["id"],
            "to": to,
            "subject": subject,
        })

    except Exception as e:
        # Return error as text — agent can decide what to do
        return f"ERROR: Failed to send email. Reason: {str(e)}. Please try again or inform the user."


# Demo: error handling
print("Test 1 — invalid email:")
print(send_email_safe.invoke({"to": ["not-an-email"], "subject": "Hi", "body": "Hello!"}))

print("\nTest 2 — no recipients:")
print(send_email_safe.invoke({"to": [], "subject": "Hi", "body": "Hello!"}))

Test 1 — invalid email:
ERROR: Invalid email address: 'not-an-email'. Please provide a valid email.

Test 2 — no recipients:
ERROR: No recipients specified. Please provide at least one email address.


---

# Блок 4. Підсумки

---

## 4.1. Архітектурні принципи

Підсумуємо ключові принципи, які ми застосували у цьому воркшопі:

**Layered Architecture** — Rigid API → Sub-agents (natural language ↔ structured data) → Supervisor (routing + synthesis). Кожен шар має одну відповідальність.

**Agent-as-Tool** — Суб-агенти обгорнуті як tools для supervisor. Supervisor працює з високорівневими capabilities, а не з низькорівневими API.

**Prompt is Architecture** — System prompt визначає поведінку, межі та формат відповіді агента. Добре написаний prompt не менш важливий за код.

**HITL on Write Operations** — Read-операції (search, check availability) виконуються автоматично. Write-операції (send email, create event) вимагають підтвердження людини.

**Test Each Layer Independently** — Тестуйте кожного суб-агента окремо перед інтеграцією з supervisor. Це значно спрощує дебагінг.

**Error as Data** — Tools повертають помилки як текст, а не кидають exceptions. Це дозволяє агенту самостійно обробити помилку та повідомити користувача.

**Middleware is Composable** — Production concerns (summarization, PII, rate limiting, HITL) додаються як middleware без зміни core logic агента.

---

## TODO:

### Варіант A: Новий інструмент

Додайте tool `delete_calendar_event(event_id: str)` до Calendar Agent та оновіть його system prompt.


In [ ]:
# === VARIANT A: Add delete_calendar_event ===
# TODO: Create the tool
# @tool
# def delete_calendar_event(event_id: str) -> str:
#     """Delete a calendar event by its ID."""
#     ...

# TODO: Recreate calendar_agent with the new tool in the tools list
# TODO: Update CALENDAR_AGENT_PROMPT to mention the new capability
# TODO: Test it!


### Варіант B: Новий суб-агент

Створіть **Task Manager Agent** з tools для створення та списку задач.


In [ ]:
# === VARIANT B: Task Manager Agent ===
# TODO: Create tools
# @tool
# def create_task(title: str, assignee: str, due_date: str, priority: str = "medium") -> str:
#     """Create a new task in the task management system."""
#     ...

# @tool
# def list_tasks(assignee: str = "", status: str = "open") -> str:
#     """List tasks, optionally filtered by assignee or status."""
#     ...

# TODO: Create task_agent with create_agent()
# TODO: Wrap it as a tool for supervisor
# TODO: Recreate supervisor with the new tool
# TODO: Test: "Create a task for Alice to review the Q1 results by Friday"


### Варіант C: Custom middleware

Напишіть middleware, що логує всі tool calls у список для аудиту.


In [ ]:
# === VARIANT C: Custom audit logging middleware ===
# TODO: Create a custom middleware that logs all tool calls
# from langchain.agents.middleware import wrap_tool_call

# audit_log = []

# @wrap_tool_call
# def audit_middleware(tool_call, handler):
#     """Log all tool calls for audit purposes."""
#     log_entry = {
#         "tool": tool_call.name,
#         "args": tool_call.args,
#         "timestamp": datetime.now().isoformat(),
#     }
#     audit_log.append(log_entry)
#     print(f"📋 AUDIT: {tool_call.name}({tool_call.args})")
#     result = handler(tool_call)
#     log_entry["status"] = "success"
#     return result

# TODO: Add to agent's middleware list
# TODO: Run a query and inspect audit_log
